## 1. Install Required Libraries

In [ ]:
# Install PyTorch 2.5.0 + CUDA 12.4
!pip install torch==2.5.0+cu124 torchvision==0.20.0+cu124 --extra-index-url https://download.pytorch.org/whl/cu124

# Install PyG dependencies
!pip install torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-2.5.0+cu124.html

# Install PyG
!pip install torch-geometric

# Install other libraries
!pip install scikit-learn matplotlib pandas

## 2. Import Libraries

In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from zipfile import ZipFile
import warnings
warnings.filterwarnings('ignore')

## 3. Upload Python Files
Upload the following 6 Python files:
- models.py
- utils.py
- dataset.py
- metrics.py
- train.py
- test_model.py

In [ ]:
from google.colab import files
uploaded = files.upload()

## 4. Upload ENZYMES Dataset
Upload the following 5 dataset files:
- ENZYMES_graph_indicator.txt
- ENZYMES_node_labels.txt
- ENZYMES_A.txt
- ENZYMES_node_attributes.txt
- ENZYMES_graph_labels.txt

In [ ]:
uploaded_data = files.upload()

## 5. Create ENZYMES Directory and Move Files

In [ ]:
!mkdir -p ENZYMES
!mv ENZYMES_*.txt ENZYMES/

## 6. Create Output Directory

In [ ]:
!mkdir -p outputs

## 7. Import Custom Modules

In [ ]:
from models import GraphClassifierGCN1, GraphClassifierGCN2, GraphClassifierGCN3
from utils import set_random_seeds, count_model_parameters
from dataset import load_enzymes_raw, split_dataset, get_data_loaders, get_dataset_info
from metrics import compute_metrics, plot_roc_curve, plot_training_curves, plot_confusion_matrix, print_classification_report
from train import train_model
from test_model import test_model

## 8. Set Random Seed for Reproducibility

In [ ]:
set_random_seeds(42)

## 9. Setup Device

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 10. Load ENZYMES Dataset

In [ ]:
dataset = load_enzymes_raw('ENZYMES')
print(f'Total graphs loaded: {len(dataset)}')

## 11. Display Dataset Information

In [ ]:
info = get_dataset_info(dataset)
print("Number of graphs:", info["num_graphs"])
print("Number of classes:", info["num_classes"])
print("Number of features:", info["num_features"])
print("Average nodes:", info["avg_nodes"])
print("Average edges:", info["avg_edges"])

## 12. Split Dataset (80/10/10)

In [ ]:
train_dataset, val_dataset, test_dataset = split_dataset(dataset, train_ratio=0.8, val_ratio=0.1, seed=42)
print(f'Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}')

## 13. Create DataLoaders

In [ ]:
batch_size = 32
train_loader, val_loader, test_loader = get_data_loaders(train_dataset, val_dataset, test_dataset, batch_size=batch_size)

## 14. Define Hyperparameters

In [ ]:
input_dim = info['num_features']
hidden_dim = 128
num_classes = info['num_classes']
dropout_rate = 0.5
learning_rate = 0.001
epochs = 100
patience = 20

print(f'Input dim: {input_dim}, Hidden dim: {hidden_dim}, Num classes: {num_classes}')

## 15. Train GCN1 Model

In [ ]:
model_gcn1 = GraphClassifierGCN1(input_dim, hidden_dim, num_classes, dropout_rate).to(device)
optimizer_gcn1 = optim.Adam(model_gcn1.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

params_gcn1 = count_model_parameters(model_gcn1)
print(f'GCN1 Parameters: {params_gcn1}')

history_gcn1, best_val_acc_gcn1 = train_model(
    model_gcn1, train_loader, val_loader, optimizer_gcn1, criterion,
    device, epochs=epochs, patience=patience, save_path='outputs/gcn1_best.pth'
)

## 16. Train GCN2 Model

In [ ]:
model_gcn2 = GraphClassifierGCN2(input_dim, hidden_dim, num_classes, dropout_rate).to(device)
optimizer_gcn2 = optim.Adam(model_gcn2.parameters(), lr=learning_rate)

params_gcn2 = count_model_parameters(model_gcn2)
print(f'GCN2 Parameters: {params_gcn2}')

history_gcn2, best_val_acc_gcn2 = train_model(
    model_gcn2, train_loader, val_loader, optimizer_gcn2, criterion,
    device, epochs=epochs, patience=patience, save_path='outputs/gcn2_best.pth'
)

## 17. Train GCN3 Model

In [ ]:
model_gcn3 = GraphClassifierGCN3(input_dim, hidden_dim, num_classes, dropout_rate).to(device)
optimizer_gcn3 = optim.Adam(model_gcn3.parameters(), lr=learning_rate)

params_gcn3 = count_model_parameters(model_gcn3)
print(f'GCN3 Parameters: {params_gcn3}')

history_gcn3, best_val_acc_gcn3 = train_model(
    model_gcn3, train_loader, val_loader, optimizer_gcn3, criterion,
    device, epochs=epochs, patience=patience, save_path='outputs/gcn3_best.pth'
)

## 18. Compare Models

In [ ]:
final_val_loss_gcn1 = history_gcn1['val_loss'][-1]
final_val_loss_gcn2 = history_gcn2['val_loss'][-1]
final_val_loss_gcn3 = history_gcn3['val_loss'][-1]

results_df = pd.DataFrame({
    'Model': ['GCN1', 'GCN2', 'GCN3'],
    'Val Accuracy': [best_val_acc_gcn1, best_val_acc_gcn2, best_val_acc_gcn3],
    'Val Loss': [final_val_loss_gcn1, final_val_loss_gcn2, final_val_loss_gcn3],
    'Params': [params_gcn1, params_gcn2, params_gcn3]
})

print(results_df)
results_df.to_csv('outputs/model_comparison.csv', index=False)

## 19. Select Best Model

In [ ]:
best_idx = results_df['Val Accuracy'].idxmax()
best_model_name = results_df.loc[best_idx, 'Model']
best_model_acc = results_df.loc[best_idx, 'Val Accuracy']

print(f'Best Model: {best_model_name} with Val Accuracy: {best_model_acc:.4f}')

if best_model_name == 'GCN1':
    best_model_class = GraphClassifierGCN1
    best_model_path = 'outputs/gcn1_best.pth'
    best_history = history_gcn1
elif best_model_name == 'GCN2':
    best_model_class = GraphClassifierGCN2
    best_model_path = 'outputs/gcn2_best.pth'
    best_history = history_gcn2
else:
    best_model_class = GraphClassifierGCN3
    best_model_path = 'outputs/gcn3_best.pth'
    best_history = history_gcn3

## 20. Test Best Model

In [ ]:
test_results = test_model(
    best_model_path, test_loader, best_model_class,
    input_dim, hidden_dim, num_classes, device, dropout_rate
)

print(f'Test Accuracy: {test_results["accuracy"]:.4f}')

test_logits = test_results['logits']
test_labels = test_results['labels']

## 21. Compute Test Metrics

In [ ]:
test_metrics = compute_metrics(test_labels, test_logits)
print(f'Test Accuracy: {test_metrics["accuracy"]:.4f}')
print(f'Test F1 Score: {test_metrics["f1_score"]:.4f}')
print(f'Test AUC: {test_metrics["auc"]:.4f}')

## 22. Plot Training Curves

In [ ]:
plot_training_curves(best_history, save_path='outputs/training_curves.png')
from IPython.display import Image, display
display(Image('outputs/training_curves.png'))

## 23. Plot ROC Curve

In [ ]:
plot_roc_curve(test_labels, test_logits, save_path='outputs/roc_curve.png')
display(Image('outputs/roc_curve.png'))

## 24. Plot Confusion Matrix

In [ ]:
class_names = [f'Class {i+1}' for i in range(num_classes)]
plot_confusion_matrix(test_labels, test_logits, class_names=class_names, save_path='outputs/confusion_matrix.png')
display(Image('outputs/confusion_matrix.png'))

## 25. Print Classification Report

In [ ]:
print_classification_report(test_labels, test_logits, class_names=class_names)

## 26. Save All Results to ZIP File

In [ ]:
with ZipFile('HW4_Results_Jamshidi.zip', 'w') as zipf:
    zipf.write('outputs/gcn1_best.pth', 'gcn1_best.pth')
    zipf.write('outputs/gcn2_best.pth', 'gcn2_best.pth')
    zipf.write('outputs/gcn3_best.pth', 'gcn3_best.pth')
    zipf.write('outputs/model_comparison.csv', 'model_comparison.csv')
    zipf.write('outputs/training_curves.png', 'training_curves.png')
    zipf.write('outputs/roc_curve.png', 'roc_curve.png')
    zipf.write('outputs/confusion_matrix.png', 'confusion_matrix.png')

print('All results saved to HW4_Results_Jamshidi.zip')

## 27. Download ZIP File

In [ ]:
files.download('HW4_Results_Jamshidi.zip')